# Multilingual Text Translator: English to Urdu and Roman Urdu Using Transformers

**Course**: Natural Language Processing (NLP)  
**Instructor**: Fawad Bahadur (Corvit)  
**Project Domain**: Neural Machine Translation (NMT) & Multilingual NLP  
**Architecture**: Sequence-to-Sequence Transformer (`Helsinki-NLP/opus-mt-en-ur`)  

---

## Project Overview & Course Alignment
This project implements an end-to-end NLP pipeline aligned with the course syllabus (`Lec 8 (NLP).pdf`):
1. **Slide 4 & 6**: Language Translation as a core NLP objective using state-of-the-art Transformer architectures.
2. **Slide 9**: Addressing multilinguality and code-switching via **Roman Urdu** representation.
3. **Slide 10-12**: Complete NLP pipeline: *Text Collection $\rightarrow$ Preprocessing $\rightarrow$ Representation $\rightarrow$ Modeling $\rightarrow$ Evaluation*.
4. **Slide 14 & 15**: Leveraging `spaCy` for linguistic analysis and `Hugging Face Transformers` for sequence modeling.
5. **Slide 17-31**: Text cleaning, contractions expansion, regex normalization, tokenization, POS tagging, and lemmatization.

### Step 0: Environment Setup & Library Installation
Run the cell below if you need to install the required libraries.

In [1]:
# Install required libraries
!pip install -q transformers torch sentencepiece sacremoses spacy sacrebleu pandas
# Download spaCy English model
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
     - ------------------------------------- 0.5/12.8 MB 316.6 kB/s eta 0:00:39
     - ------------------------------------- 0.5/12.8 MB

### Imports & Environment Verification

In [2]:
import re
import torch
import pandas as pd
import spacy
from transformers import MarianMTModel, MarianTokenizer
import sacrebleu

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch Device: {device}")
print(f"PyTorch Version: {torch.__version__}")

c:\Users\Mushtaq Ahmad Madni\miniconda3\envs\HCCDA\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch Device: cpu
PyTorch Version: 2.14.1+cpu


---
## Step 1: Raw Text Input and Informal English Normalization

**Lecture Reference**: 
- **Slides 27-28 (Text Normalization)**: Expanding contractions (`don't` $\rightarrow$ `do not`, `I'm` $\rightarrow$ `I am`) and handling conversational slang (`gonna` $\rightarrow$ `going to`).
- **Slides 29-31 (Regular Expressions)**: Pattern matching with Python `re` module to strip unwanted web noise and normalize spacing.

In [3]:
# 1. Contractions and Informal English Mapping (Lecture Slides 27-28)
CONTRACTIONS_MAP = {
    r"\bi'm\b": "I am",
    r"\bi've\b": "I have",
    r"\bi'll\b": "I will",
    r"\bi'd\b": "I would",
    r"\byou're\b": "you are",
    r"\byou've\b": "you have",
    r"\byou'll\b": "you will",
    r"\bhe's\b": "he is",
    r"\bshe's\b": "she is",
    r"\bit's\b": "it is",
    r"\bwe're\b": "we are",
    r"\bthey're\b": "they are",
    r"\bdon't\b": "do not",
    r"\bdoesn't\b": "does not",
    r"\bdidn't\b": "did not",
    r"\bcan't\b": "cannot",
    r"\bwon't\b": "will not",
    r"\bgonna\b": "going to",
    r"\bwanna\b": "want to",
    r"\bgotta\b": "got to",
    r"\blol\b": "laughing out loud",
    r"\bpls\b": "please",
    r"\bplz\b": "please",
    r"\bu\b": "you",
    r"\br\b": "are"
}

def expand_contractions(text: str) -> str:
    for pattern, replacement in CONTRACTIONS_MAP.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
    return text

def clean_noise_regex(text: str) -> str:
    # Remove HTML tags & URLs
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    # Normalize multiple whitespace
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def normalize_english_input(text: str) -> str:
    text = expand_contractions(text)
    text = clean_noise_regex(text)
    return text

# Demonstration of Step 1 Normalization
sample_raw_inputs = [
    "I'm gonna learn artificial intelligence today, lol!",
    "Don't worry, u r doing great. Plz visit http://nlp.org <br>",
    "I can't wait to see my new university."
]

print("=== Step 1: Normalization Results ===")
for raw in sample_raw_inputs:
    cleaned = normalize_english_input(raw)
    print(f"Raw    : {raw}")
    print(f"Cleaned: {cleaned}\n")

=== Step 1: Normalization Results ===
Raw    : I'm gonna learn artificial intelligence today, lol!
Cleaned: I am going to learn artificial intelligence today, laughing out loud!

Raw    : Don't worry, u r doing great. Plz visit http://nlp.org <br>
Cleaned: do not worry, you are doing great. please visit

Raw    : I can't wait to see my new university.
Cleaned: I cannot wait to see my new university.



---
## Step 2: spaCy Demonstration (Tokenization, POS Tagging, Lemmatization)

**Lecture Reference**:
- **Slide 14**: spaCy as a fast, production-ready NLP library.
- **Slides 19-21**: Tokenization (Splitting text into words/sentences).
- **Slides 22-24**: Lemmatization vs. Stemming (Grammar-aware base forms vs. naive suffix truncation).
- **Slides 25-26**: Stop Words (Why stop words like *'not'* must NOT be removed in translation).

In [4]:
try:
    nlp = spacy.load("en_core_web_sm")
except Exception:
    import spacy.cli
    spacy.cli.download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

demo_sentence = "The dedicated students are studying artificial intelligence in the classroom."
doc = nlp(demo_sentence)

# 1. Word Tokenization, POS Tagging, and Lemmatization Table
spacy_records = []
for token in doc:
    spacy_records.append({
        "Token": token.text,
        "Lemma (Root)": token.lemma_,
        "POS Tag": token.pos_,
        "Detailed Tag": token.tag_,
        "Is Stopword?": token.is_stop
    })

df_spacy = pd.DataFrame(spacy_records)
print("=== Step 2: spaCy Linguistic Analysis Table ===")
print(df_spacy.to_string(index=False))

print("\n--- Key Viva Point on Machine Translation Preprocessing ---")
print("Notice that words like 'are' and 'not' are classified as stopwords.")
print("In classification, we often remove them. But for Machine Translation,")
print("removing stopwords or using aggressive stemming destroys sentence meaning and tense!")
print("Hence, our translation pipeline retains grammatical structure.")

=== Step 2: spaCy Linguistic Analysis Table ===
       Token Lemma (Root) POS Tag Detailed Tag  Is Stopword?
         The          the     DET           DT          True
   dedicated    dedicated     ADJ           JJ         False
    students      student    NOUN          NNS         False
         are           be     AUX          VBP          True
    studying        study    VERB          VBG         False
  artificial   artificial     ADJ           JJ         False
intelligence intelligence    NOUN           NN         False
          in           in     ADP           IN          True
         the          the     DET           DT          True
   classroom    classroom    NOUN           NN         False
           .            .   PUNCT            .         False

--- Key Viva Point on Machine Translation Preprocessing ---
Notice that words like 'are' and 'not' are classified as stopwords.
In classification, we often remove them. But for Machine Translation,
removing stopwords or

---
## Step 3: Hugging Face Transformer Model (English to Urdu Translation)

**Lecture Reference**:
- **Slide 6**: 2018-Present Transformers & LLMs (Vaswani et al. 2017) with transfer learning.
- **Slide 15**: Hugging Face `transformers` library for state-of-the-art Sequence-to-Sequence models.

We use **`Helsinki-NLP/opus-mt-en-ur`**, a dedicated MarianMT sequence-to-sequence Transformer trained specifically for English-to-Urdu translation.

In [5]:
MODEL_NAME = "Helsinki-NLP/opus-mt-en-ur"

print(f"Loading Pretrained Transformer: {MODEL_NAME}...")
tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)
model = MarianMTModel.from_pretrained(MODEL_NAME).to(device)
model.eval()
print("Model loaded successfully!")

def translate_english_to_urdu(text: str, max_length: int = 128) -> str:
    # 1. Preprocess input safely (expand contractions & clean noise)
    cleaned_input = normalize_english_input(text)
    if not cleaned_input:
        return ""
    
    # 2. Tokenize input with Marian subword tokenizer
    inputs = tokenizer(cleaned_input, return_tensors="pt", padding=True, truncation=True, max_length=max_length)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # 3. Generate target tokens using beam search
    with torch.no_grad():
        translated_tokens = model.generate(
            **inputs,
            num_beams=4,
            max_length=max_length,
            early_stopping=True
        )
    
    # 4. Decode tokens back to Urdu script
    urdu_text = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)
    return urdu_text

# Test translation with various English sentences
test_sentences = [
    "How are you?",
    "I want to learn artificial intelligence.",
    "This is a great natural language processing project.",
    "I don't like wasting time."
]

print("\n=== Step 3: English to Urdu Translation Samples ===")
for sent in test_sentences:
    urdu_trans = translate_english_to_urdu(sent)
    print(f"English: {sent}")
    print(f"Urdu   : {urdu_trans}\n")

Loading Pretrained Transformer: Helsinki-NLP/opus-mt-en-ur...


c:\Users\Mushtaq Ahmad Madni\miniconda3\envs\HCCDA\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Mushtaq Ahmad Madni\.cache\huggingface\hub\models--Helsinki-NLP--opus-mt-en-ur. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\Mushtaq Ahmad Madni\miniconda3\envs\HCCDA\Lib\site-packages\huggingf

Model loaded successfully!

=== Step 3: English to Urdu Translation Samples ===
English: How are you?
Urdu   : آپ کیسے ہیں؟

English: I want to learn artificial intelligence.
Urdu   : میں مصنوعی ذہانت سیکھنا چاہتا ہوں.

English: This is a great natural language processing project.
Urdu   : یہ ایک قدرتی زبان کا ایک بہت بڑا پروجیکٹ ہے.

English: I don't like wasting time.
Urdu   : مجھے وقت ضائع کرنے کی طرح نہیں لگتا.



---
## Step 4: Urdu to Roman Urdu Conversion (Transliteration)

**Lecture Reference**:
- **Slide 9 (Multilinguality & Code-Switching)**: In digital communication across Pakistan and South Asia, users frequently communicate using Roman script (e.g., *"Main office jaa raha hoon"*).

Here we implement a hybrid transliterator: combining a high-frequency Urdu lexicon for natural Roman spelling with rule-based phonetic fallback.

In [6]:
# Import our custom transliteration module
from transliteration import urdu_to_roman

print("=== Step 4: End-to-End Multilingual Translation (English -> Urdu -> Roman Urdu) ===\n")

samples_for_roman = [
    "How are you?",
    "I want to learn artificial intelligence.",
    "Welcome to our university.",
    "I am going to school with my friend.",
    "Thank you very much."
]

results = []
for eng in samples_for_roman:
    urdu_script = translate_english_to_urdu(eng)
    roman_urdu = urdu_to_roman(urdu_script)
    results.append({
        "English Input": eng,
        "Urdu Script": urdu_script,
        "Roman Urdu": roman_urdu
    })

df_results = pd.DataFrame(results)
for i, row in df_results.iterrows():
    print(f"[{i+1}] English   : {row['English Input']}")
    print(f"    Urdu      : {row['Urdu Script']}")
    print(f"    Roman Urdu: {row['Roman Urdu']}\n")

=== Step 4: End-to-End Multilingual Translation (English -> Urdu -> Roman Urdu) ===

[1] English   : How are you?
    Urdu      : آپ کیسے ہیں؟
    Roman Urdu: Aap kaise hin??

[2] English   : I want to learn artificial intelligence.
    Urdu      : میں مصنوعی ذہانت سیکھنا چاہتا ہوں.
    Roman Urdu: Main masnooi zehanat seekhna chahta hoon

[3] English   : Welcome to our university.
    Urdu      : میری یونیورسٹی میں خوش آمدید
    Roman Urdu: Miri university main khush aamdeed

[4] English   : I am going to school with my friend.
    Urdu      : میں اپنے دوست کے ساتھ اسکول جا رہا ہوں.
    Roman Urdu: Main apne dost ke saath askol ja raha hoon

[5] English   : Thank you very much.
    Urdu      : بہت بہت شکریہ.
    Roman Urdu: Bohat bohat shukriya



---
## Step 5: Test Dataset Evaluation using SacreBLEU & chrF

**Lecture Reference**:
- **Slide 12 (Evaluation)**: Measuring the quality of machine-generated text using standard objective metrics.

In Machine Translation, **BLEU** (Bilingual Evaluation Understudy) measures n-gram precision against human reference translations, while **chrF** evaluates character n-gram F-score (ideal for morphologically rich languages like Urdu).

In [7]:
# Benchmark evaluation dataset of parallel English-Urdu pairs
eval_data = [
    {
        "english": "How are you?",
        "reference_urdu": "آپ کیسے ہیں؟"
    },
    {
        "english": "I want to learn artificial intelligence.",
        "reference_urdu": "میں مصنوعی ذہانت سیکھنا چاہتا ہوں۔"
    },
    {
        "english": "This is my book.",
        "reference_urdu": "یہ میری کتاب ہے۔"
    },
    {
        "english": "I am very happy today.",
        "reference_urdu": "میں آج بہت خوش ہوں۔"
    },
    {
        "english": "Where are you going?",
        "reference_urdu": "آپ کہاں جا رہے ہیں؟"
    },
    {
        "english": "Education is very important for everyone.",
        "reference_urdu": "تعلیم ہر ایک کے لئے بہت اہم ہے۔"
    }
]

# Run model translations
hypotheses = []
references = []
eval_records = []

for item in eval_data:
    pred_urdu = translate_english_to_urdu(item["english"])
    hypotheses.append(pred_urdu)
    references.append(item["reference_urdu"])
    eval_records.append({
        "English": item["english"],
        "Reference Urdu": item["reference_urdu"],
        "Model Prediction": pred_urdu,
        "Roman Urdu": urdu_to_roman(pred_urdu)
    })

# Compute BLEU & chrF scores using sacrebleu
# Note: references must be a list of lists of references
bleu = sacrebleu.corpus_bleu(hypotheses, [[r for r in references]])
chrf = sacrebleu.corpus_chrf(hypotheses, [[r for r in references]])

df_eval = pd.DataFrame(eval_records)
print("=== Evaluation Dataset & Predictions ===")
print(df_eval[["English", "Reference Urdu", "Model Prediction"]].to_string(index=False))

print("\n=== Benchmark Evaluation Metrics ===")
print(f"Corpus BLEU Score : {bleu.score:.2f} / 100")
print(f"Corpus chrF Score : {chrf.score:.2f} / 100")
print("\nInterpretation:")
print("- BLEU > 30 represents high-quality readable translations in Machine Translation benchmarks.")
print("- chrF evaluates character matches, demonstrating robustness to Urdu inflection and spelling variations.")

=== Evaluation Dataset & Predictions ===
                                  English                     Reference Urdu                   Model Prediction
                             How are you?                       آپ کیسے ہیں؟                       آپ کیسے ہیں؟
 I want to learn artificial intelligence. میں مصنوعی ذہانت سیکھنا چاہتا ہوں۔ میں مصنوعی ذہانت سیکھنا چاہتا ہوں.
                         This is my book.                   یہ میری کتاب ہے۔                   یہ میری کتاب ہے.
                   I am very happy today.                میں آج بہت خوش ہوں۔              مَیں آج بہت خوش ہوں ۔
                     Where are you going?                آپ کہاں جا رہے ہیں؟                 تم کہاں جا رہے ہو؟
Education is very important for everyone.    تعلیم ہر ایک کے لئے بہت اہم ہے۔     تعلیم سب کے لئے بہت ضروری ہے ۔

=== Benchmark Evaluation Metrics ===
Corpus BLEU Score : 34.42 / 100
Corpus chrF Score : 77.63 / 100

Interpretation:
- BLEU > 30 represents high-quality readable translation

---
## Conclusion & Course Project Summary

1. **Pipeline Completeness**: We implemented every stage of the classical and modern NLP pipeline (Text Preprocessing $\rightarrow$ Linguistic Feature Extraction with spaCy $\rightarrow$ Pretrained Transformer Translation $\rightarrow$ Roman Urdu Transliteration $\rightarrow$ Corpus BLEU/chrF Evaluation).
2. **Real-world Multilingual Impact**: Solves the critical South Asian linguistic challenge where users alternate between Urdu script and Roman Urdu (code-switching).
3. **Deployment**: This pipeline can now be served interactively via the companion Streamlit web application (`app.py`).